In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd
from notebookutils import mssparkutils  # Use mssparkutils if notebookutils isn't default

StatementMeta(, bd280fac-53bf-4ee5-aebf-3088381000b8, 3, Finished, Available, Finished, False)

#### Load Single File

In [16]:
#df_sales= pd.read_excel("abfss://Development_Workspace@onelake.dfs.fabric.microsoft.com/Lakehouse_Bronze.Lakehouse/Files/Sales_01012023.xlsx",sheet_name="Sales")

# display(df_sales)

StatementMeta(, f7b78bb5-c296-4395-bf8c-bdf4e7c9691a, 22, Finished, Available, Finished, False)

#### Load All Files in directory 

In [2]:
#import pandas as pd
#from notebookutils import mssparkutils  # Use mssparkutils if notebookutils isn't default

# 1. Define the base directory path
base_dir = "abfss://Development_Workspace@onelake.dfs.fabric.microsoft.com/Lakehouse_Bronze.Lakehouse/Files/"

# 2. List all files in the directory dynamically
all_files = mssparkutils.fs.ls(base_dir)

# 3. Filter for files that match your pattern (e.g., starting with 'Sales_')
sales_files = [
    file.path for file in all_files 
    if file.name.startswith("Sales_") and file.name.endswith(".xlsx")
]

# 4. Read and concatenate all files into a single DataFrame
df_list = []
for file_path in sales_files:
    df = pd.read_excel(file_path, sheet_name="Sales")
    df_list.append(df)

# Final combined DataFrame
df_sales = pd.concat(df_list, ignore_index=True)


StatementMeta(, bd280fac-53bf-4ee5-aebf-3088381000b8, 5, Finished, Available, Finished, False)

In [3]:
df1=spark.createDataFrame(df_sales)

StatementMeta(, bd280fac-53bf-4ee5-aebf-3088381000b8, 6, Finished, Available, Finished, False)

In [4]:
display(df1.head(10))

StatementMeta(, bd280fac-53bf-4ee5-aebf-3088381000b8, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2aeb264d-cd21-4336-8ba0-b58094503b0c)

In [19]:
# df_Returns= pd.read_excel("abfss://Development_Workspace@onelake.dfs.fabric.microsoft.com/Lakehouse_Bronze.Lakehouse/Files/Sales_01012023.xlsx",sheet_name="Returns")

StatementMeta(, f7b78bb5-c296-4395-bf8c-bdf4e7c9691a, 26, Finished, Available, Finished, False)

In [5]:
# For Ruturn 

# 4. Read and concatenate all files into a single DataFrame
df_list = []
for file_path in sales_files:
    df = pd.read_excel(file_path, sheet_name="Returns")
    df_list.append(df)

# Final combined DataFrame
df_Returns = pd.concat(df_list, ignore_index=True)

StatementMeta(, bd280fac-53bf-4ee5-aebf-3088381000b8, 9, Finished, Available, Finished, False)

In [6]:
df2=spark.createDataFrame(df_Returns)

StatementMeta(, bd280fac-53bf-4ee5-aebf-3088381000b8, 10, Finished, Available, Finished, False)

In [7]:
display(df2.head(10))

StatementMeta(, bd280fac-53bf-4ee5-aebf-3088381000b8, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6f82f05c-b9ab-405d-ba7d-9c363e6a05f2)

In [8]:
df_final=df1.join(df2,df1.Order_ID==df2.Order_ID,how="left").drop(df2.Order_ID,df2.Customer_Name,df2.Sales_Amount)

StatementMeta(, bd280fac-53bf-4ee5-aebf-3088381000b8, 13, Finished, Available, Finished, False)

In [9]:
df_mod=df_final.withColumns({"Order_Year": year("Order_Date"), \
                "Order_Month": month("Order_Date"), \
                "Created_TS": current_timestamp(),\
                "Modified_TS": current_timestamp(),\
                })

StatementMeta(, bd280fac-53bf-4ee5-aebf-3088381000b8, 14, Finished, Available, Finished, False)

In [10]:
df_mod.createOrReplaceTempView("ViewSales")

StatementMeta(, bd280fac-53bf-4ee5-aebf-3088381000b8, 15, Finished, Available, Finished, False)

In [11]:
%%sql
create table if not exists Lakehouse_Bronze.dbo.Bronze_Sales
(
    Order_ID	string,
    Order_Date	Date,
    Shipping_Date	date,
    Aging	int,
    Ship_Mode	string,
    Product_Category	string,
    Product	 string,
    Sales 	float,
    Quantity	float,
    Discount	 float,
    Profit 	 float,
    Shipping_Cost 	float,
    Order_Priority	string,
    Customer_ID	string,
    Customer_Name	string,
    Segment	string,
    City	string,
    State	string,
    Country	string,
    Region string,
	Return string,
    Order_Year int,
    Order_Month int,
    Created_TS  TIMESTAMP,
    Modified_TS  TIMESTAMP
)
using DELTA
PARTITIONED by(Order_Year,Order_Month)



StatementMeta(, bd280fac-53bf-4ee5-aebf-3088381000b8, 16, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [12]:
%%sql
Merge into  Lakehouse_Bronze.dbo.Bronze_Sales as BS
using ViewSales as VS
on  BS.Order_Year=VS.Order_Year and BS.Order_Month=VS.Order_Month and BS.Order_ID=VS.Order_ID
when matched then 
update SET
BS.Order_Date	=	VS.Order_Date	,
BS.Shipping_Date	=	VS.Shipping_Date	,
BS.Aging	=	VS.Aging	,
BS.Ship_Mode	=	VS.Ship_Mode	,
BS.Product_Category	=	VS.Product_Category	,
BS.Product	=	VS.Product	,
BS.Sales 	=	VS.Sales 	,
BS.Quantity	=	VS.Quantity	,
BS.Discount	=	VS.Discount	,
BS.Profit 	=	VS.Profit 	,
BS.Shipping_Cost 	=	VS.Shipping_Cost 	,
BS.Order_Priority	=	VS.Order_Priority	,
BS.Customer_ID	=	VS.Customer_ID	,
BS.Customer_Name	=	VS.Customer_Name	,
BS.Segment	=	VS.Segment	,
BS.City	=	VS.City	,
BS.State	=	VS.State	,
BS.Country	=	VS.Country	,
BS.Region  	=	VS.Region  	,
BS.Return  	=	VS.Return  	,
BS.Modified_TS	=	VS.Modified_TS	

when not matched then 
INSERT
(
BS.Order_ID,    
BS.Order_Date	,
BS.Shipping_Date	,
BS.Aging	,
BS.Ship_Mode	,
BS.Product_Category	,
BS.Product	,
BS.Sales 	,
BS.Quantity	,
BS.Discount	,
BS.Profit 	,
BS.Shipping_Cost 	,
BS.Order_Priority	,
BS.Customer_ID	,
BS.Customer_Name	,
BS.Segment	,
BS.City	,
BS.State	,
BS.Country	,
BS.Region  	,
BS.Return,
BS.Order_Year  	,
BS.Order_Month  	,
BS.Created_TS   	,
BS.Modified_TS	
)
values
(
VS.Order_ID,
VS.Order_Date	,
VS.Shipping_Date	,
VS.Aging	,
VS.Ship_Mode	,
VS.Product_Category	,
VS.Product	,
VS.Sales 	,
VS.Quantity	,
VS.Discount	,
VS.Profit 	,
VS.Shipping_Cost 	,
VS.Order_Priority	,
VS.Customer_ID	,
VS.Customer_Name	,
VS.Segment	,
VS.City	,
VS.State	,
VS.Country	,
VS.Region  	,
VS.Return,
VS.Order_Year  	,
VS.Order_Month  	,
VS.Created_TS   	,
VS.Modified_TS	

)


StatementMeta(, bd280fac-53bf-4ee5-aebf-3088381000b8, 17, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>